In [1]:
%store -r

In [ ]:
import functools
import glob
import json
import math
import os
import subprocess

import commute_dm.queries
import commute_dm.utils
import credentials
import momapy.celldesigner.core
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session
import pandas

In [ ]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

In [3]:
def get_species_true_class_names():
    classes_with_subclasses = set([])
    classes = set([])
    for attr_name in dir(momapy.celldesigner.core):
        attr_value = getattr(momapy.celldesigner.core, attr_name)
        if isinstance(attr_value, type):
            classes.add(attr_value)
            for base_class in attr_value.__bases__:
                classes_with_subclasses.add(base_class)
    classes_with_no_subclass = classes.difference(classes_with_subclasses)
    return [class_.__name__ for class_ in classes_with_no_subclass]

In [4]:
class_name_to_bel_namespaces = {
    "Drug": [
        "drugbank",
        "obo.chebi",
        "pubchem.compound",
        "mesh",
        "ncit",
    ],
    "GenericProtein": [
        "uniprot",
        "pr",
        "interpro",
        "mesh",
        "ncbiprotein",  # to check with Heval and Alpha
    ],
    "Receptor": [
        "uniprot",
        "pr",
        "interpro",
        "mesh",
        "ncbiprotein",  # to check with Heval and Alpha
    ],
    "IonChannel": [
        "uniprot",
        "pr",
        "interpro",
        "mesh",
        "ncbiprotein",  # to check with Heval and Alpha
    ],
    "TruncatedProtein": [
        "uniprot",
        "pr",
        "interpro",
        "mesh",
        "ncbiprotein",  # to check with Heval and Alpha
    ],
    "SimpleMolecule": [
        "obo.chebi",
        "pubchem.compound",
        "mesh",
        "ncit",
    ],
    "Ion": [
        "obo.chebi",
        "pubchem.compound",
        "mesh",
        "ncit",
    ],
    "Phenotype": [
        "hp",
        "snomedct",
        "mesh",
        "obo.go",
        "wikipathways",
        "reactome",
        "doid",
        "mondo",
        "mesh",
        "omim",  # to check with Heval and Alpha
    ],
    "Gene": [
        "hgnc.symbol",
        "mgi",
        "rgd",
        "ncbigene",
    ],
    "RNA": [
        "hgnc.symbol",
        "mgi",
        "rgd",
        "ncbigene",
        "refseq",  # to check with Heval and Alpha
    ],
    "Complex": [
        "obo.go",
        "pr",
        "mesh",
        "cl",
        "clo",
        "uniprot",  # to check with Heval and Alpha
        "ncbiprotein",  # to check with Heval and Alpha
    ],
    "Compartment": ["cl", "clo", "obo.go", "mesh", "uberon", "ncit"],
    "Degraded": [],
}

In [ ]:
query = """MATCH (species:Species) WHERE NOT EXISTS ((species)-[:HAS_SUBUNIT]->()) AND NOT species:Degraded RETURN species"""
results = session.execute_query(query)
species = [row["species"] for row in results]
annotations = commute_dm.queries.get_annotations(session, species)

In [7]:
avoid_namespaces = set(["taxonomy", "pubmed", "doi"])
already_uris = {}
for species, species_annotations in annotations:
    class_name = next(
        iter(species.labels.intersection(class_name_to_bel_namespaces.keys()))
    )
    name = species["name"].replace("\n", " ")
    key = (
        name,
        class_name,
    )
    if key not in already_uris:
        already_uris[key] = set([])
    for species_annotation in species_annotations:
        resources = tuple(
            [
                resource
                for resource in species_annotation["resources"]
                if resource.split(":")[2] not in avoid_namespaces
            ]
        )
        if resources:
            already_uris[key].add(resources)

In [8]:
df = pandas.read_csv("annotated_full_non_empty_nano.csv", sep="\t")

In [9]:
uris = []
for _, row in df.iterrows():
    if row["prediction correct"] == "y":
        id_ = str(row["id (predicted)"])
        if ":" not in id_:
            uri = f"{row['db'].lower()}:{id_}"
        else:
            uri = id_
    else:
        uri = str(row["uri (manual)"])
        if uri != "nan":
            uri = ";".join(
                [
                    f"{_.split(':')[0].lower().replace(' ', '')}:{_.split(':')[1]}"
                    for _ in uri.split(";")
                ]
            )
        else:
            uri = ""
    uris.append(uri)
df["uri"] = uris

In [10]:
new_uris = {}
new_comments = {}
for _, row in df.iterrows():
    name, class_name, uris, comments = (
        row["name"],
        row["type"],
        row["uri"],
        row["comment (uri manual)"],
    )
    uris = tuple(uris.split(";"))
    if (name, class_name) not in new_uris:
        new_uris[(name, class_name)] = set([])
        new_comments[(name, class_name)] = comments
    new_uris[(name, class_name)].add(uris)

In [11]:
df = pandas.read_csv("manually_annotated.csv", sep="\t")

In [12]:
for _, row in df.iterrows():
    name, class_name, uris, comments = (
        row["label"],
        row["type"],
        row["identifiers"],
        row["comment"],
    )
    if isinstance(uris, str):
        uris = tuple([_.replace(" ", "") for _ in uris.split(";")])
    else:
        uris = tuple([])
    if class_name == "Protein":
        class_names = ["GenericProtein", "TruncatedProtein", "Receptor", "Ion channel"]
    else:
        class_names = [class_name]
    for class_name in class_names:
        if (name, class_name) not in new_uris:
            new_uris[(name, class_name)] = set([])
        new_uris[(name, class_name)].add(uris)
        new_comments[(name, class_name)] = comments

In [ ]:
results = []
for species, species_annotations in annotations:
    class_name = next(
        iter(species.labels.intersection(class_name_to_bel_namespaces.keys()))
    )
    name = species["name"].replace("\n", " ")
    key = (
        name,
        class_name,
    )
    uris = set([])
    uris_no_publications = set([])
    for species_annotation in species_annotations:
        resources = tuple(species_annotation["resources"])
        if resources:
            uris.add(resources)
        resources_no_publications = tuple(
            [
                resource
                for resource in species_annotation["resources"]
                if resource.split(":")[2] not in avoid_namespaces
            ]
        )
        if resources_no_publications:
            uris_no_publications.add(resources_no_publications)
    if not uris_no_publications:
        if already_uris[key]:
            to_add = already_uris[key].difference(uris_no_publications)
            from_ = "existent"
            comments = ""
        else:
            if key in new_uris:
                to_add = new_uris[key].difference(uris_no_publications)
                from_ = "new"
                comments = new_comments[key]
            else:
                to_add = []
                from_ = "absent"
        ids_and_context = commute_dm.queries.get_ids_and_context(session, [species])
        if (
            not comments
            or isinstance(comments, float)
            and math.isnan(comments)
            or comments == "nan"
        ):
            comments = ""
        for record in ids_and_context:
            for id_, entry_node, collection_node in record[1]:
                results.append(
                    {
                        "collection": collection_node["name"],
                        "map": entry_node["id_"],
                        "entity_id": id_,
                        "entity_name": name,
                        "additional_annotations": [list(_) for _ in to_add],
                        "from": from_,
                        "comments": comments,
                    }
                )

In [14]:
print(len(results))

519


In [15]:
with open("results_annotation.json", "w") as f:
    json.dump(results, f, indent=4)